In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# (F)(G) 設定
# =========================
SEED = 42
np.random.seed(SEED)

base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
out_root = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
model_name_dir = "RandomForest"
output_dir = os.path.join(out_root, f"Corr_1000_{model_name_dir}")
os.makedirs(output_dir, exist_ok=True)

file_names = [
    "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
    "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
    "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
    "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
]

target_vars = ["Jsc", "Voc", "FF", "PCEmax"]
# (D)(E) 除外変数
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'DRMS', 'Var.1'
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

# =========================
# Helpers
# =========================
def safe_r2(y_true, y_pred) -> float:
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return float("nan")

def remove_collinear_features(df: pd.DataFrame, cols, threshold=0.99999):
    if len(cols) < 2:
        return cols
    corr = df[cols].corr().abs().fillna(0.0)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    return [c for c in cols if c not in to_drop]

# =========================
# Main
# =========================
summary_rows = []
importance_rows = []

for file in file_names:
    in_path = os.path.join(base_path, file)
    if not os.path.exists(in_path):
        continue

    print(f"\n=== Processing Random Forest: {file} ===")
    df_raw = pd.read_csv(in_path, index_col=0).drop(columns=["X"], errors="ignore")
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()

    if df_num.shape[0] < 20:
        continue

    for target in target_vars:
        if target not in df_num.columns:
            continue

        # --- (C)(D)(E) 特徴量選択 ---
        feature_cols = [c for c in df_num.columns if c not in target_vars]
        feature_cols = [c for c in feature_cols if c not in INAPPROPRIATE_VARS]
        # 正規表現による除外
        for pat in PHYSICAL_EXCLUDE_PATTERNS:
            feature_cols = [c for c in feature_cols if pat not in c]

        # (H) 多重共線性・ゼロ分散除外
        v = df_num[feature_cols].var()
        feature_cols = v[v > 0].index.tolist()
        feature_cols = remove_collinear_features(df_num, feature_cols)

        if len(feature_cols) == 0:
            continue

        # (I) Scaling (-1, 1)
        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_scaled = scaler.fit_transform(df_num[feature_cols])
        y = df_num[target]

        # --- Hyperparameter tuning ---
        param_grid = {
            "n_estimators": [200, 500],
            "max_depth": [None, 20, 40],
            "min_samples_leaf": [1, 3, 5],
            "max_features": ["sqrt", 0.3, 0.5]
        }
        cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
        grid = GridSearchCV(
            estimator=RandomForestRegressor(random_state=SEED, n_jobs=-1),
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_scaled, y)
        best_model = grid.best_estimator_

        # --- (K) CV10 Out-of-Fold 予測 ---
        y_oof = cross_val_predict(best_model, X_scaled, y, cv=cv)
        
        cv_r2 = safe_r2(y, y_oof)
        cv_rmse = float(np.sqrt(mean_squared_error(y, y_oof)))
        cv_mae = float(mean_absolute_error(y, y_oof))

        # --- 結果保存 ---
        file_tag = file.replace('.csv', '')
        
        # 1. モデル保存
        joblib.dump(best_model, os.path.join(output_dir, f"{file_tag}_{target}_model.joblib"))

        # 2. (B)(K) CV10_OOF 予測CSV
        df_pred_oof = pd.DataFrame({
            "SampleID": df_num.index.astype(str),
            "Observed": y.to_numpy(),
            "Predicted": y_oof.astype(float),
            "Type": "CV10_OOF"
        })
        df_pred_oof.to_csv(os.path.join(output_dir, f"{file_tag}_{target}_pred_OOF.csv"), index=False)

        # 3. 重要度 (Gini-based)
        importances = best_model.feature_importances_
        for idx, val in enumerate(importances):
            if val > 0:
                importance_rows.append({
                    "File": file, "Target": target,
                    "Feature": feature_cols[idx], "Importance": val
                })

        # 4. サマリー
        summary_rows.append({
            "File": file, "Target": target,
            "CV10_R2": cv_r2, "CV10_RMSE": cv_rmse, "CV10_MAE": cv_mae,
            "n_samples": int(df_num.shape[0]), "n_features": int(len(feature_cols)),
            "Best_Params": json.dumps(grid.best_params_)
        })

        print(f"  Target={target} | CV10_R2={cv_r2:.3f} | Best_Params={grid.best_params_}")

# CSV保存
pd.DataFrame(summary_rows).to_csv(os.path.join(output_dir, "all_summary_CV10.csv"), index=False)
pd.DataFrame(importance_rows).to_csv(os.path.join(output_dir, "all_importance_RF.csv"), index=False)
print("\n[DONE] RandomForest Analysis completed.")



=== Processing Random Forest: n_base.csv ===
  Target=Jsc | CV10_R2=0.707 | Best_Params={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 500}
  Target=Voc | CV10_R2=0.815 | Best_Params={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 500}
  Target=FF | CV10_R2=0.490 | Best_Params={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
  Target=PCEmax | CV10_R2=0.663 | Best_Params={'max_depth': 20, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 500}

=== Processing Random Forest: n_base_OH_rebuilt.csv ===
  Target=Jsc | CV10_R2=0.770 | Best_Params={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
  Target=Voc | CV10_R2=0.818 | Best_Params={'max_depth': 40, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
  Target=FF | CV10_R2=0.521 | Best_Params={'max_depth': 40, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
  T

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# (F)(G) 設定
# =========================
SEED = 42
np.random.seed(SEED)

base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
out_root = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
model_name_dir = "RandomForest"
output_dir = os.path.join(out_root, f"Corr_1000_{model_name_dir}")
os.makedirs(output_dir, exist_ok=True)

file_names = [
"m_base_FP_rebuilt.csv"

]

target_vars = ["Voc"]
# (D)(E) 除外変数
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'DRMS', 'Var.1'
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

# =========================
# Helpers
# =========================
def safe_r2(y_true, y_pred) -> float:
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return float("nan")

def remove_collinear_features(df: pd.DataFrame, cols, threshold=0.99999):
    if len(cols) < 2:
        return cols
    corr = df[cols].corr().abs().fillna(0.0)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    return [c for c in cols if c not in to_drop]

# =========================
# Main
# =========================
summary_rows = []
importance_rows = []

for file in file_names:
    in_path = os.path.join(base_path, file)
    if not os.path.exists(in_path):
        continue

    print(f"\n=== Processing Random Forest: {file} ===")
    df_raw = pd.read_csv(in_path, index_col=0).drop(columns=["X"], errors="ignore")
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()

    if df_num.shape[0] < 20:
        continue

    for target in target_vars:
        if target not in df_num.columns:
            continue

        # --- (C)(D)(E) 特徴量選択 ---
        feature_cols = [c for c in df_num.columns if c not in target_vars]
        feature_cols = [c for c in feature_cols if c not in INAPPROPRIATE_VARS]
        # 正規表現による除外
        for pat in PHYSICAL_EXCLUDE_PATTERNS:
            feature_cols = [c for c in feature_cols if pat not in c]

        # (H) 多重共線性・ゼロ分散除外
        v = df_num[feature_cols].var()
        feature_cols = v[v > 0].index.tolist()
        feature_cols = remove_collinear_features(df_num, feature_cols)

        if len(feature_cols) == 0:
            continue

        # (I) Scaling (-1, 1)
        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_scaled = scaler.fit_transform(df_num[feature_cols])
        y = df_num[target]

        # --- Hyperparameter tuning ---
        param_grid = {
            "n_estimators": [200, 500],
            "max_depth": [None, 20, 40],
            "min_samples_leaf": [1, 3, 5],
            "max_features": ["sqrt", 0.3, 0.5]
        }
        cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
        grid = GridSearchCV(
            estimator=RandomForestRegressor(random_state=SEED, n_jobs=-1),
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_scaled, y)
        best_model = grid.best_estimator_

        # --- (K) CV10 Out-of-Fold 予測 ---
        y_oof = cross_val_predict(best_model, X_scaled, y, cv=cv)
        
        cv_r2 = safe_r2(y, y_oof)
        cv_rmse = float(np.sqrt(mean_squared_error(y, y_oof)))
        cv_mae = float(mean_absolute_error(y, y_oof))

        # --- 結果保存 ---
        file_tag = file.replace('.csv', '')
        
        # 1. モデル保存
        joblib.dump(best_model, os.path.join(output_dir, f"{file_tag}_{target}_model.joblib"))

        # 2. (B)(K) CV10_OOF 予測CSV
        df_pred_oof = pd.DataFrame({
            "SampleID": df_num.index.astype(str),
            "Observed": y.to_numpy(),
            "Predicted": y_oof.astype(float),
            "Type": "CV10_OOF"
        })
        df_pred_oof.to_csv(os.path.join(output_dir, f"{file_tag}_{target}_pred_OOF.csv"), index=False)

        # 3. 重要度 (Gini-based)
        importances = best_model.feature_importances_
        for idx, val in enumerate(importances):
            if val > 0:
                importance_rows.append({
                    "File": file, "Target": target,
                    "Feature": feature_cols[idx], "Importance": val
                })

        # 4. サマリー
        summary_rows.append({
            "File": file, "Target": target,
            "CV10_R2": cv_r2, "CV10_RMSE": cv_rmse, "CV10_MAE": cv_mae,
            "n_samples": int(df_num.shape[0]), "n_features": int(len(feature_cols)),
            "Best_Params": json.dumps(grid.best_params_)
        })

        print(f"  Target={target} | CV10_R2={cv_r2:.3f} | Best_Params={grid.best_params_}")

# CSV保存
pd.DataFrame(summary_rows).to_csv(os.path.join(output_dir, "all_summary_CV10.csv"), index=False)
pd.DataFrame(importance_rows).to_csv(os.path.join(output_dir, "all_importance_RF.csv"), index=False)
print("\n[DONE] RandomForest Analysis completed.")
